### Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModel, pipeline
import string
from datetime import datetime

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("punkt")
nltk.download("stopwords")

STOPWORDS = set(stopwords.words("english"))
vader = SentimentIntensityAnalyzer()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\02_Florian_Benutzer\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\02_Florian_Benutzer\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [ ]:
# import pandas as pd
# import numpy as np
# import re

# from transformers import pipeline, AutoTokenizer, AutoModel
# import torch

# CSV-Daten laden

Donald Trump Tweets bis 2021

In [2]:
tweets = pd.read_csv("C:/Users/02_Florian_Benutzer/Desktop/trump_oil_analysis_v2/00_data/trump_tweets.csv")

In [3]:
tweets.head()

,id,text,is_retweet,is_deleted,device,favorites,retweets,datetime,is_flagged,date
0,9.845497e+16,Republicans and Democrats have both created ou...,False,False,TweetDeck,49,255,2011-08-02T18:07:48Z,False,2011-08-02
1,1.234653e+18,I was thrilled to be back in the Great city of...,False,False,Twitter for iPhone,73748,17404,2020-03-03T01:34:50Z,False,2020-03-03
2,1.218011e+18,RT @CBS_Herridge: READ: Letter to surveillance...,True,False,Twitter for iPhone,0,7396,2020-01-17T03:22:47Z,False,2020-01-17
3,1.304875e+18,The Unsolicited Mail In Ballot Scam is a major...,False,False,Twitter for iPhone,80527,23502,2020-09-12T20:10:58Z,False,2020-09-12
4,1.218160e+18,RT @MZHemingway: Very friendly telling of even...,True,False,Twitter for iPhone,0,9081,2020-01-17T13:13:59Z,False,2020-01-17


In [4]:
tweets.shape

(56571, 10)

In [5]:
tweets_v1 = tweets.copy()

In [6]:
# 2. Den String in ein "naives" (zeitzonenloses) Datetime-Objekt umwandeln
# (Pandas erkennt das Format meistens automatisch)
tweets_v1['timestamp_utc'] = pd.to_datetime(tweets_v1['datetime'])

# 4. In UTC umrechnen
#tweets_v1['timestamp_utc'] = tweets_v1['timestamp'].dt.tz_localize('UTC')

# 5. (Optional) Wenn du das "+00:00" am Ende der UTC-Zeit nicht in der CSV haben willst,
# kannst du die Zeitzonen-Information wieder entfernen (die Zeit bleibt die von UTC):
#tweets_v1['timestamp_utc_clean'] = tweets_v1['timestamp_utc'].dt.tz_localize(None)

# Ergebnis überprüfen
#print(df[['Timestamp', 'Timestamp_UTC_clean']].head())

In [7]:
tweets_v1.dtypes

id                           float64
text                             str
is_retweet                      bool
is_deleted                      bool
device                           str
favorites                      int64
retweets                       int64
datetime                         str
is_flagged                      bool
date                             str
timestamp_utc    datetime64[us, UTC]
dtype: object

In [8]:
tweets_v1.shape

(56571, 11)

In [9]:
tweets_v1.head()

,id,text,is_retweet,is_deleted,device,favorites,retweets,datetime,is_flagged,date,timestamp_utc
0,9.845497e+16,Republicans and Democrats have both created ou...,False,False,TweetDeck,49,255,2011-08-02T18:07:48Z,False,2011-08-02,2011-08-02 18:07:48+00:00
1,1.234653e+18,I was thrilled to be back in the Great city of...,False,False,Twitter for iPhone,73748,17404,2020-03-03T01:34:50Z,False,2020-03-03,2020-03-03 01:34:50+00:00
2,1.218011e+18,RT @CBS_Herridge: READ: Letter to surveillance...,True,False,Twitter for iPhone,0,7396,2020-01-17T03:22:47Z,False,2020-01-17,2020-01-17 03:22:47+00:00
3,1.304875e+18,The Unsolicited Mail In Ballot Scam is a major...,False,False,Twitter for iPhone,80527,23502,2020-09-12T20:10:58Z,False,2020-09-12,2020-09-12 20:10:58+00:00
4,1.218160e+18,RT @MZHemingway: Very friendly telling of even...,True,False,Twitter for iPhone,0,9081,2020-01-17T13:13:59Z,False,2020-01-17,2020-01-17 13:13:59+00:00


Retweets werden aus der Analyse ausgeschlossen

In [10]:
tweets_v2 = tweets_v1.copy()

In [11]:
tweets_v2 = tweets_v2[tweets_v2['is_retweet'] == False] 

In [12]:
tweets_v2.shape

(46694, 11)

Die Spalten 'is_deleted', 'favorites', 'retweets' und 'is_flagged' werden entfernt um Data Leakage zu vermeiden

In [13]:
tweets_v2 = tweets_v2.drop(columns=['is_retweet', 'is_deleted', 'device', 'favorites', 'retweets', 'is_flagged'])

In [14]:
tweets_v2.shape

(46694, 5)

In [15]:
tweets_v2

,id,text,datetime,date,timestamp_utc
0,9.845497e+16,Republicans and Democrats have both created ou...,2011-08-02T18:07:48Z,2011-08-02,2011-08-02 18:07:48+00:00
1,1.234653e+18,I was thrilled to be back in the Great city of...,2020-03-03T01:34:50Z,2020-03-03,2020-03-03 01:34:50+00:00
3,1.304875e+18,The Unsolicited Mail In Ballot Scam is a major...,2020-09-12T20:10:58Z,2020-09-12,2020-09-12 20:10:58+00:00
6,1.223641e+18,Getting a little exercise this morning! https:...,2020-02-01T16:14:02Z,2020-02-01,2020-02-01 16:14:02+00:00
7,1.319502e+18,https://t.co/4qwCKQOiOw,2020-10-23T04:52:14Z,2020-10-23,2020-10-23 04:52:14+00:00
...,...,...,...,...,...
56555,1.213079e+18,"Iran never won a war, but never lost a negotia...",2020-01-03T12:44:30Z,2020-01-03,2020-01-03 12:44:30+00:00
56559,1.212177e+18,Thank you to the @dcexaminer Washington Examin...,2020-01-01T01:03:15Z,2020-01-01,2020-01-01 01:03:15+00:00
56560,1.212175e+18,One of my greatest honors was to have gotten C...,2020-01-01T00:55:01Z,2020-01-01,2020-01-01 00:55:01+00:00
56569,1.319384e+18,Just signed an order to support the workers of...,2020-10-22T21:04:21Z,2020-10-22,2020-10-22 21:04:21+00:00


In [16]:
tweets_v2.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v2.csv", sep=",", index=False, encoding="utf-8")

# Feature Engineering

### 1. Imports

In [1]:
import re
import warnings

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from transformers import pipeline

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from bertopic import BERTopic

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v2.csv")

### 2. Configuration

In [3]:
TEXT_COL = "text"
TIMESTAMP_COL = "timestamp_utc"

SENTENCE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

PCA_COMPONENTS = 20
N_CLUSTERS = 10

TEST_SIZE = 0.20

### 3. Input

In [4]:
tweets_df = df.copy()

tweets_df[TIMESTAMP_COL] = pd.to_datetime(
    tweets_df[TIMESTAMP_COL],
    utc=True
)

tweets_df = tweets_df.sort_values(
    TIMESTAMP_COL
).reset_index(drop=True)

### 4. Cleaning

In [5]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)

    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


tweets_df["clean_text"] = (
    tweets_df[TEXT_COL]
    .fillna("")
    .apply(clean_text)
)

### 5. Models

In [35]:
embed_model = SentenceTransformer(
    SENTENCE_MODEL
)

finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    return_all_scores=True,
    top_k=None
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### 6. Embeddings

In [7]:
embeddings = embed_model.encode(
    tweets_df["clean_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.asarray(embeddings)

tweets_df["embedding"] = list(
    embeddings
)

Batches:   0%|          | 0/1460 [00:00<?, ?it/s]

### 7. FinBERT

In [40]:
def extract_finbert_features(text):

    try:

        scores = finbert(
            text,
            truncation=True,
            max_length=512
        )[0]

        scores = {
            s["label"].lower(): s["score"]
            for s in scores
        }

        pos = scores.get("positive", 0)
        neu = scores.get("neutral", 0)
        neg = scores.get("negative", 0)

        # kleine numerische Stabilität
        eps = 1e-10

        entropy = (
            -(pos * np.log(pos + eps)
              + neu * np.log(neu + eps)
              + neg * np.log(neg + eps))
        )

        confidence = max(pos, neu, neg)

        return pd.Series({

            # Original probabilities
            "finbert_positive": pos,
            "finbert_neutral": neu,
            "finbert_negative": neg,

            # Sentiment signal
            "finbert_sentiment_score": pos - neg,

            # Uncertainty / structure
            "finbert_entropy": entropy,
            "finbert_confidence": confidence,

            # your original risk/uncertainty definitions
            "financial_uncertainty_score": 1 - confidence,
            "financial_risk_sentiment": neg,
            "finbert_polarization": abs(pos - neg)
        })

    except Exception as e:
        print(e)

        return pd.Series({

            "finbert_positive": np.nan,
            "finbert_neutral": np.nan,
            "finbert_negative": np.nan,

            "finbert_sentiment_score": np.nan,
            "finbert_entropy": np.nan,
            "finbert_confidence": np.nan,

            "financial_uncertainty_score": np.nan,
            "financial_risk_sentiment": np.nan,
            "finbert_polarization": np.nan
        })

In [14]:
tweets_df = pd.concat(
    [
        tweets_df,
        tweets_df["clean_text"]
        .apply(extract_finbert_features)
    ],
    axis=1
)

In [15]:
tweets_df.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v3.csv", sep=",", index=False, encoding="utf-8")

In [ ]:
# def extract_finbert_features(text):

#     try:

#         scores = finbert(
#             text,
#             truncation=True,
#             max_length=512
#         )[0]

#         scores = {
#             s["label"].lower(): s["score"]
#             for s in scores
#         }

#         pos = scores.get(
#             "positive", 0
#         )

#         neu = scores.get(
#             "neutral", 0
#         )

#         neg = scores.get(
#             "negative", 0
#         )

#         return pd.Series({

#             "finbert_sentiment_score":
#                 pos - neg,

#             "financial_uncertainty_score":
#                 1 - pos,

#             "financial_risk_sentiment":
#                 neg,

#             "finbert_positive":
#                 pos,

#             "finbert_neutral":
#                 neu,

#             "finbert_negative":
#                 neg
#         })

#     except Exception:

#         return pd.Series({

#             "finbert_sentiment_score": np.nan,
#             "financial_uncertainty_score": np.nan,
#             "financial_risk_sentiment": np.nan,
#             "finbert_positive": np.nan,
#             "finbert_neutral": np.nan,
#             "finbert_negative": np.nan
#         })


# tweets_df = pd.concat(
#     [
#         tweets_df,
#         tweets_df["clean_text"]
#         .apply(extract_finbert_features)
#     ],
#     axis=1
# )

KeyboardInterrupt: 

### 8. Semantic Centroids

In [ ]:
reference_texts = {

    "wti":
    "oil supply shock geopolitics OPEC crude war sanctions energy inflation",

    "geo":
    "war sanctions Russia Iran Middle East conflict military escalation",

    "energy":
    "oil production OPEC drilling supply demand crude inventory",

    "macro":
    "inflation interest rates GDP recession monetary policy Fed",

    "policy":
    "policy decision executive order sanctions trade war"
}

reference_embeddings = {

    name: embed_model.encode(
        text,
        normalize_embeddings=True
    )

    for name, text
    in reference_texts.items()
}

### 9. Similarity Features

In [ ]:
def similarity_to_reference(
    emb,
    ref
):

    return cosine_similarity(
        emb.reshape(1, -1),
        ref.reshape(1, -1)
    )[0, 0]


for name, ref in reference_embeddings.items():

    tweets_df[f"{name}_similarity"] = [

        similarity_to_reference(
            emb,
            ref
        )

        for emb in embeddings
    ]


tweets_df.rename(
    columns={
        "wti_similarity":
            "wti_relevance_score",

        "geo_similarity":
            "geopolitical_similarity_score",

        "energy_similarity":
            "energy_supply_similarity_score",

        "macro_similarity":
            "macro_economy_similarity_score",

        "policy_similarity":
            "policy_strength_score"
    },
    inplace=True
)

### 10. Linguistic Features

In [ ]:
def caps_ratio(text):

    words = str(text).split()

    if len(words) == 0:
        return 0

    return (
        sum(
            w.isupper()
            for w in words
        )
        / len(words)
    )


tweets_df["caps_ratio"] = (
    tweets_df[TEXT_COL]
    .fillna("")
    .apply(caps_ratio)
)

tweets_df["exclamation_count"] = (
    tweets_df[TEXT_COL]
    .fillna("")
    .str.count("!")
)

tweets_df["tweet_length"] = (
    tweets_df["clean_text"]
    .str.split()
    .apply(len)
)

tweets_df["market_aggression_index"] = (

      0.5 * tweets_df["caps_ratio"]
    + 0.3 * tweets_df["exclamation_count"]
    + 0.2 * tweets_df["tweet_length"]

)

### 11. Temporal Features

In [ ]:
tweets_df["time_since_last_tweet_min"] = (

    tweets_df[TIMESTAMP_COL]
    .diff()
    .dt.total_seconds()
    .div(60)

)

tweets_df[
    "time_since_last_tweet_min"
] = (
    tweets_df[
        "time_since_last_tweet_min"
    ].fillna(0)
)

### 12. Leakage-Safe Rolling Features

In [ ]:
tweets_df = tweets_df.set_index(
    TIMESTAMP_COL
)

tweets_df["rolling_tweet_frequency_60m"] = (

    pd.Series(
        1,
        index=tweets_df.index
    )
    .rolling("60min")
    .sum()
    .shift(1)
)

tweets_df["rolling_tweet_frequency_6h"] = (

    pd.Series(
        1,
        index=tweets_df.index
    )
    .rolling("6h")
    .sum()
    .shift(1)
)

tweets_df["tweet_burst_indicator"] = (
    tweets_df[
        "rolling_tweet_frequency_60m"
    ] > 3
).astype(int)

### 13. Sentimen History

In [ ]:
tweets_df["sentiment_delta_vs_previous"] = (

    tweets_df[
        "finbert_sentiment_score"
    ]
    -
    tweets_df[
        "finbert_sentiment_score"
    ].shift(1)

)

tweets_df["rolling_sentiment_mean_60m"] = (

    tweets_df[
        "finbert_sentiment_score"
    ]
    .rolling("60min")
    .mean()
    .shift(1)

)

tweets_df["rolling_sentiment_std_60m"] = (

    tweets_df[
        "finbert_sentiment_score"
    ]
    .rolling("60min")
    .std()
    .shift(1)

)

### 14. Novelty

In [ ]:
novelty = [0]

for i in range(
    1,
    len(embeddings)
):

    sim = cosine_similarity(
        embeddings[i].reshape(1,-1),
        embeddings[:i]
    ).max()

    novelty.append(
        1 - sim
    )

tweets_df["novelty_score"] = novelty

### 15. Semantic Distance

In [ ]:
distance_last = [0]

for i in range(
    1,
    len(embeddings)
):

    sim = cosine_similarity(
        embeddings[i].reshape(1,-1),
        embeddings[i-1].reshape(1,-1)
    )[0,0]

    distance_last.append(
        1 - sim
    )

tweets_df[
    "semantic_distance_to_last_tweet"
] = distance_last

### 16. Interaction Features

In [ ]:
tweets_df["sentiment_x_geopolitics"] = (

    tweets_df[
        "finbert_sentiment_score"
    ]
    *
    tweets_df[
        "geopolitical_similarity_score"
    ]
)

tweets_df["arousal_x_oil_relevance"] = (

    tweets_df[
        "market_aggression_index"
    ]
    *
    tweets_df[
        "wti_relevance_score"
    ]
)

### 17. Market Shock Index

In [ ]:
tweets_df["tweet_market_shock_index"] = (

      0.25 * tweets_df["finbert_sentiment_score"]
    + 0.20 * tweets_df["market_aggression_index"]
    + 0.20 * tweets_df["geopolitical_similarity_score"]
    + 0.15 * tweets_df["novelty_score"]
    + 0.20 * tweets_df["wti_relevance_score"]
)

### 18. Time Split

In [ ]:
tweets_df = tweets_df.reset_index()

split_time = tweets_df[
    TIMESTAMP_COL
].quantile(
    1 - TEST_SIZE
)

train_df = tweets_df[
    tweets_df[TIMESTAMP_COL]
    <= split_time
].copy()

test_df = tweets_df[
    tweets_df[TIMESTAMP_COL]
    > split_time
].copy()

### 19. PCA (Leakage Safe)

In [ ]:
train_embeddings = np.vstack(
    train_df["embedding"]
)

test_embeddings = np.vstack(
    test_df["embedding"]
)

scaler = StandardScaler()

train_embeddings_scaled = (
    scaler.fit_transform(
        train_embeddings
    )
)

test_embeddings_scaled = (
    scaler.transform(
        test_embeddings
    )
)

pca = PCA(
    n_components=PCA_COMPONENTS,
    random_state=42
)

train_pca = pca.fit_transform(
    train_embeddings_scaled
)

test_pca = pca.transform(
    test_embeddings_scaled
)

### 20. PCA Features

In [ ]:
for i in range(
    PCA_COMPONENTS
):

    train_df[
        f"embedding_pca_{i}"
    ] = train_pca[:, i]

    test_df[
        f"embedding_pca_{i}"
    ] = test_pca[:, i]

### 21. BERTopic

In [ ]:
topic_model = BERTopic(
    verbose=True
)

train_topics, _ = (
    topic_model.fit_transform(
        train_df["clean_text"]
        .tolist()
    )
)

test_topics, _ = (
    topic_model.transform(
        test_df["clean_text"]
        .tolist()
    )
)

train_df["topic_id"] = (
    train_topics
)

test_df["topic_id"] = (
    test_topics
)

### 22. KMeans

In [ ]:
kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=42,
    n_init="auto"
)

kmeans.fit(train_pca)

train_df["cluster_id"] = (
    kmeans.predict(train_pca)
)

test_df["cluster_id"] = (
    kmeans.predict(test_pca)
)

### 23. Final Cleanup

In [ ]:
final_df = pd.concat(
    [train_df, test_df]
)

final_df = final_df.sort_values(
    TIMESTAMP_COL
)

final_df = final_df.replace(
    [np.inf, -np.inf],
    np.nan
)

final_df = final_df.fillna(0)

### 24. Export

In [ ]:
final_df.to_parquet(
    "tweet_feature_store.parquet",
    index=False
)

final_df.to_csv(
    "tweet_feature_store.csv",
    index=False
)

### 24. Export

# Alter Ansatz folgt

### 1. Imports

In [42]:
import numpy as np
import pandas as pd

from datetime import timedelta

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

import re

In [44]:
import numpy as np
import pandas as pd

import re

from transformers import pipeline
from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

### 2. Input

In [43]:
tweets_df = df.copy()

tweets_df["timestamp"] = pd.to_datetime(
    tweets_df["timestamp_utc"],
    utc=True
)

tweets_df = tweets_df.sort_values("timestamp")

TEXT_COL = "text"

### 3. Cleaning

In [45]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()


tweets_df["clean_text"] = tweets_df[TEXT_COL].apply(clean_text)

### 4. Sentence Embeddings

In [46]:
embed_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embed_model.encode(
    tweets_df["clean_text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.array(embeddings)

tweets_df["embedding"] = list(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1460 [00:00<?, ?it/s]

### 5. FinBERT Sentiment

In [47]:
finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    return_all_scores=True
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [48]:
def get_finbert_score(text):

    try:

        scores = finbert(text[:512])[0]

        scores = {
            s["label"].lower(): s["score"]
            for s in scores
        }

        positive = scores["positive"]
        neutral = scores["neutral"]
        negative = scores["negative"]

        sentiment = (
            positive
            - negative
        )

        return sentiment

    except:

        return np.nan

In [ ]:
tweets_df["finbert_sentiment_score"] = (
    tweets_df["clean_text"]
    .apply(get_finbert_score)
)

### 6. Semantic Reference Centroids

In [ ]:
reference_texts = {

    "wti":
    "oil supply shock geopolitics OPEC crude war sanctions energy inflation",

    "geo":
    "war sanctions Russia Iran Middle East conflict military escalation",

    "energy":
    "oil production OPEC drilling supply demand crude inventory",

    "macro":
    "inflation interest rates GDP recession monetary policy Fed",

    "policy":
    "policy decision executive order sanctions trade war"
}

In [ ]:
reference_embeddings = {}

for name, text in reference_texts.items():

    reference_embeddings[name] = (
        embed_model.encode(
            text,
            normalize_embeddings=True
        )
    )

### 7. Similarity Features

In [ ]:
def cosine_to_reference(vec, ref):

    return cosine_similarity(
        vec.reshape(1,-1),
        ref.reshape(1,-1)
    )[0,0]

In [ ]:
tweets_df["wti_relevance_score"] = [
    cosine_to_reference(
        emb,
        reference_embeddings["wti"]
    )
    for emb in embeddings
]

tweets_df["geopolitical_similarity_score"] = [
    cosine_to_reference(
        emb,
        reference_embeddings["geo"]
    )
    for emb in embeddings
]

tweets_df["energy_supply_similarity_score"] = [
    cosine_to_reference(
        emb,
        reference_embeddings["energy"]
    )
    for emb in embeddings
]

tweets_df["macro_economy_similarity_score"] = [
    cosine_to_reference(
        emb,
        reference_embeddings["macro"]
    )
    for emb in embeddings
]

tweets_df["policy_strength_score"] = [
    cosine_to_reference(
        emb,
        reference_embeddings["policy"]
    )
    for emb in embeddings
]

### 8. Linguistic Features

In [ ]:
def caps_ratio(text):

    words = text.split()

    if len(words) == 0:
        return 0

    upper = sum(
        w.isupper()
        for w in words
    )

    return upper / len(words)

In [ ]:
tweets_df["caps_ratio"] = (
    tweets_df[TEXT_COL]
    .fillna("")
    .apply(caps_ratio)
)

tweets_df["exclamation_count"] = (
    tweets_df[TEXT_COL]
    .fillna("")
    .str.count("!")
)

tweets_df["tweet_length"] = (
    tweets_df["clean_text"]
    .str.split()
    .apply(len)
)

In [ ]:
tweets_df["market_aggression_index"] = (
      0.5 * tweets_df["caps_ratio"]
    + 0.3 * tweets_df["exclamation_count"]
    + 0.2 * tweets_df["tweet_length"]
)

### 9. Temporal Features

In [ ]:
tweets_df["time_since_last_tweet_min"] = (

    tweets_df["timestamp"]
    .diff()
    .dt.total_seconds()
    / 60

)

tweets_df["time_since_last_tweet_min"] = (
    tweets_df["time_since_last_tweet_min"]
    .fillna(np.nan)
)

### 10. Rolling Tweet Frequency

In [ ]:
tweets_df = tweets_df.set_index("timestamp")

In [ ]:
tweets_df["rolling_tweet_frequency_60m"] = (
    pd.Series(
        1,
        index=tweets_df.index
    )
    .rolling("60min")
    .sum()
    .shift(1)
)

tweets_df["rolling_tweet_frequency_6h"] = (
    pd.Series(
        1,
        index=tweets_df.index
    )
    .rolling("6h")
    .sum()
    .shift(1)
)

In [ ]:
tweets_df["tweet_burst_indicator"] = (
    tweets_df["rolling_tweet_frequency_60m"]
    > 3
).astype(int)

### 11. Sentiment History

In [ ]:
tweets_df["sentiment_delta_vs_previous"] = (
    tweets_df["finbert_sentiment_score"]
    -
    tweets_df["finbert_sentiment_score"].shift(1)
)

In [ ]:
tweets_df["rolling_sentiment_mean_60m"] = (
    tweets_df["finbert_sentiment_score"]
    .rolling("60min")
    .mean()
    .shift(1)
)

tweets_df["rolling_sentiment_std_60m"] = (
    tweets_df["finbert_sentiment_score"]
    .rolling("60min")
    .std()
    .shift(1)
)

### 12. NoveltyFeatures

In [ ]:
novelty = [np.nan]

for i in range(1, len(embeddings)):

    sims = cosine_similarity(
        embeddings[i].reshape(1,-1),
        embeddings[:i]
    )

    novelty.append(
        1 - sims.max()
    )

tweets_df["novelty_score"] = novelty

In [ ]:
distance_last = [np.nan]

for i in range(1, len(embeddings)):

    sim = cosine_similarity(
        embeddings[i].reshape(1,-1),
        embeddings[i-1].reshape(1,-1)
    )[0,0]

    distance_last.append(
        1 - sim
    )

tweets_df["semantic_distance_to_last_tweet"] = (
    distance_last
)

### 13. Interaction Features

In [ ]:
tweets_df["sentiment_x_geopolitics"] = (
    tweets_df["finbert_sentiment_score"]
    *
    tweets_df["geopolitical_similarity_score"]
)

tweets_df["uncertainty_x_macro"] = (
    np.abs(
        tweets_df["finbert_sentiment_score"]
    )
    *
    tweets_df["macro_economy_similarity_score"]
)

### 14. Market Shock Index

In [ ]:
tweets_df["tweet_market_shock_index"] = (

      0.25 * tweets_df["finbert_sentiment_score"]

    + 0.20 * tweets_df["geopolitical_similarity_score"]

    + 0.20 * tweets_df["novelty_score"]

    + 0.15 * tweets_df["caps_ratio"]

    + 0.20 * tweets_df["wti_relevance_score"]

)

### 15. Zeitbasierter Split 

In [ ]:
tweets_df = tweets_df.reset_index()

In [ ]:
train_end = "2018-12-31"

train_df = (
    tweets_df[
        tweets_df["timestamp"] <= train_end
    ]
)

test_df = (
    tweets_df[
        tweets_df["timestamp"] > train_end
    ]
)

### 16. ERST JETZT: PC/BERTopic/KMeans

In [ ]:
pca.fit(train_embeddings)

train_pca = pca.transform(train_embeddings)

test_pca = pca.transform(test_embeddings)

In [ ]:
topic_model.fit(train_texts)

test_topics = topic_model.transform(test_texts)

In [ ]:
kmeans.fit(train_embeddings)

test_cluster = kmeans.predict(test_embeddings)

# Trennung: nachfolgend nur noch aktuell verforfener Code der ggf nochal relevant wird

In [ ]:
import numpy as np
import pandas as pd

from datetime import timedelta

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

import re

In [ ]:
# =========================
# USER SETTINGS
# =========================

tweets_df = YOUR_TWITTER_DF  # <- CHANGE THIS ONLY

tweets_df = tweets_df.copy()

tweets_df["timestamp"] = pd.to_datetime(tweets_df["timestamp"], utc=True)
tweets_df = tweets_df.sort_values("timestamp")

TEXT_COL = "text"

# embedding model names (optional swap)
SENTENCE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

In [ ]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer

# FinBERT sentiment
finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    return_all_scores=True
)

# general embeddings
embed_model = SentenceTransformer(SENTENCE_MODEL)

In [ ]:
def clean_text(t):
    if pd.isna(t):
        return ""
    t = t.lower()
    t = re.sub(r"http\S+", "", t)
    t = re.sub(r"@\w+", "", t)
    t = re.sub(r"#", "", t)
    t = re.sub(r"[^a-zA-Z\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

tweets_df["clean_text"] = tweets_df[TEXT_COL].apply(clean_text)

In [ ]:
embeddings = embed_model.encode(
    tweets_df["clean_text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.array(embeddings)

tweets_df["embedding"] = list(embeddings)

In [ ]:
def finbert_score(text):
    try:
        res = finbert(text[:512])[0]
        scores = {r["label"]: r["score"] for r in res}

        pos = scores.get("positive", 0)
        neg = scores.get("negative", 0)
        neu = scores.get("neutral", 0)

        # expectation value
        return pos * 1 + neu * 0 + neg * (-1)

    except:
        return 0.0


def finbert_uncertainty(text):
    try:
        res = finbert(text[:512])[0]
        scores = {r["label"]: r["score"] for r in res}
        return 1 - scores.get("positive", 0)
    except:
        return 0.0


def finbert_risk(text):
    try:
        res = finbert(text[:512])[0]
        scores = {r["label"]: r["score"] for r in res}
        return scores.get("negative", 0)
    except:
        return 0.0

In [ ]:
tweets_df["finbert_sentiment_score"] = tweets_df["clean_text"].apply(finbert_score)
tweets_df["financial_uncertainty_score"] = tweets_df["clean_text"].apply(finbert_uncertainty)
tweets_df["financial_risk_sentiment"] = tweets_df["clean_text"].apply(finbert_risk)

In [ ]:
def centroid_embedding(texts):
    vecs = embed_model.encode(texts, normalize_embeddings=True)
    return np.mean(vecs, axis=0)


oil_centroid = centroid_embedding([
    "oil supply shock geopolitics OPEC crude war sanctions energy inflation"
])

geo_centroid = centroid_embedding([
    "war sanctions Russia Iran Middle East conflict military escalation"
])

energy_centroid = centroid_embedding([
    "oil production OPEC drilling supply demand crude inventory"
])

macro_centroid = centroid_embedding([
    "inflation interest rates GDP recession monetary policy Fed"
])

In [ ]:
def cos_sim(a, b):
    return cosine_similarity([a], [b])[0][0]

In [ ]:
tweets_df["wti_relevance_score"] = embeddings.tolist()
tweets_df["geopolitical_similarity_score"] = embeddings.tolist()
tweets_df["energy_supply_similarity_score"] = embeddings.tolist()
tweets_df["macro_economy_similarity_score"] = embeddings.tolist()

tweets_df["wti_relevance_score"] = tweets_df["wti_relevance_score"].apply(lambda x: cos_sim(x, oil_centroid))
tweets_df["geopolitical_similarity_score"] = tweets_df["geopolitical_similarity_score"].apply(lambda x: cos_sim(x, geo_centroid))
tweets_df["energy_supply_similarity_score"] = tweets_df["energy_supply_similarity_score"].apply(lambda x: cos_sim(x, energy_centroid))
tweets_df["macro_economy_similarity_score"] = tweets_df["macro_economy_similarity_score"].apply(lambda x: cos_sim(x, macro_centroid))

In [ ]:
def caps_ratio(text):
    if not text:
        return 0
    words = text.split()
    if len(words) == 0:
        return 0
    caps = sum([w.isupper() for w in words])
    return caps / len(words)


tweets_df["caps_ratio"] = tweets_df[TEXT_COL].apply(caps_ratio)
tweets_df["exclamation_count"] = tweets_df[TEXT_COL].str.count("!")
tweets_df["tweet_length"] = tweets_df["clean_text"].str.split().apply(len)

tweets_df["market_aggression_index"] = (
    0.5 * tweets_df["caps_ratio"] +
    0.3 * tweets_df["exclamation_count"] +
    0.2 * tweets_df["tweet_length"]
)

In [ ]:
tweets_df = tweets_df.sort_values("timestamp")

tweets_df["time_since_last_tweet_min"] = (
    tweets_df["timestamp"] - tweets_df["timestamp"].shift(1)
).dt.total_seconds() / 60

tweets_df["time_since_last_tweet_min"] = tweets_df["time_since_last_tweet_min"].fillna(0)

In [ ]:
def rolling_count(df, minutes):
    out = []
    for i, t in enumerate(df["timestamp"]):
        past = df.iloc[:i]
        window_start = t - timedelta(minutes=minutes)
        out.append(((past["timestamp"] >= window_start) & (past["timestamp"] < t)).sum())
    return np.array(out)

In [ ]:
tweets_df["rolling_tweet_frequency_60m"] = rolling_count(tweets_df, 60)
tweets_df["rolling_tweet_frequency_6h"] = rolling_count(tweets_df, 360)

tweets_df["tweet_burst_indicator"] = (tweets_df["rolling_tweet_frequency_60m"] > 3).astype(int)

In [ ]:
tweets_df["sentiment_delta_vs_previous"] = (
    tweets_df["finbert_sentiment_score"] -
    tweets_df["finbert_sentiment_score"].shift(1)
).fillna(0)

In [ ]:
tweets_df["rolling_sentiment_mean_60m"] = rolling_count(tweets_df, 60) * 0.0
tweets_df["rolling_sentiment_std_60m"] = rolling_count(tweets_df, 60) * 0.0

In [ ]:
def semantic_novelty(i):
    if i == 0:
        return 0
    sim = cosine_similarity(
        [embeddings[i]],
        embeddings[:i]
    ).max()
    return 1 - sim


tweets_df["novelty_score"] = [semantic_novelty(i) for i in range(len(tweets_df))]

In [ ]:
def semantic_distance_to_last(i):
    if i == 0:
        return 0
    return 1 - cosine_similarity(
        [embeddings[i]],
        [embeddings[i-1]]
    )[0][0]


tweets_df["semantic_distance_to_last_tweet"] = [
    semantic_distance_to_last(i) for i in range(len(tweets_df))
]

In [ ]:
tweets_df["sentiment_x_geopolitics"] = (
    tweets_df["finbert_sentiment_score"] *
    tweets_df["geopolitical_similarity_score"]
)

tweets_df["arousal_x_oil_relevance"] = (
    tweets_df["market_aggression_index"] *
    tweets_df["wti_relevance_score"]
)

In [ ]:
tweets_df["tweet_market_shock_index"] = (
    0.25 * tweets_df["finbert_sentiment_score"] +
    0.20 * tweets_df["market_aggression_index"] +
    0.20 * tweets_df["geopolitical_similarity_score"] +
    0.15 * tweets_df["novelty_score"] +
    0.20 * tweets_df["wti_relevance_score"]
)

In [ ]:
pca = PCA(n_components=20)

embedding_pca = pca.fit_transform(embeddings)

for i in range(embedding_pca.shape[1]):
    tweets_df[f"embedding_pca_{i}"] = embedding_pca[:, i]

In [ ]:
split_time = tweets_df["timestamp"].quantile(0.8)

train_df = tweets_df[tweets_df["timestamp"] <= split_time]
test_df  = tweets_df[tweets_df["timestamp"] > split_time]

# Rest

In [19]:
df.shape

(46694, 5)

### Tweets von Whitespaces und URLS bereinigen

In [20]:
def clean_text(text):
#    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)  # URLs
#    text = re.sub(r"@\w+", " USER ", text)  # mentions
#    text = re.sub(r"#", "", text)  # hashtags symbol
#    text = re.sub(r"\d+", " NUM ", text)
#    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [21]:
df["clean_text"] = df["text"].astype(str).apply(clean_text)

### Temporale Features

In [25]:
df["timestamp"] = pd.to_datetime(df["timestamp_utc"], utc=True)

df["minute"] = df["timestamp"].dt.minute
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month
df["year"] = df["timestamp"].dt.year
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

### Hashtag / Mention Features

In [27]:
def social_features(text):
    return pd.Series({
        "hashtag_count": text.count("#"),
        "mention_count": text.count("@"),
        "url_count": len(re.findall(r"http\S+", text))
    })

In [28]:
df[["hashtag_count", "mention_count", "url_count"]] = df["text"].apply(social_features)

### Text- & Sprachstil-Features

In [ ]:
def count_all_caps_words(text: str) -> int:
    return len(re.findall(r'\b[A-ZÄÖÜ]{2,}\b', text))


def count_repeated_words(text: str) -> int:
    words = re.findall(r'\b\w+\b', text.lower())
    return sum(words[i] == words[i + 1] for i in range(len(words) - 1))


def avg_word_length(text: str) -> float:
    words = re.findall(r'\b\w+\b', text)
    return np.mean([len(w) for w in words]) if words else 0.0


def count_sentences(text: str) -> int:
    sentences = re.split(r'[.!?]+', text)
    return len([s for s in sentences if s.strip()])


def count_pattern_words(text: str, pattern_list) -> int:
    text_lower = text.lower()
    return sum(len(re.findall(r'\b' + re.escape(p) + r'\b', text_lower)) for p in pattern_list)


def capital_ratio(text: str) -> float:
    """
    Anteil Großbuchstaben an allen alphabetischen Zeichen.
    """
    letters = re.findall(r'[A-Za-z]', text)
    if not letters:
        return 0.0
    caps = sum(1 for c in letters if c.isupper())
    return caps / len(letters)


# def negation_count(text: str) -> int:
#     """
#     Zählt typische Negationswörter.
#     """
#     negations = [
#         "not", "no", "never", "nothing", "none", "nobody",
#         "don't", "doesn't", "didn't", "won't", "can't", "cannot",
#         "isn't", "aren't", "wasn't", "weren't", "n't"
#     ]
    
#     text_lower = text.lower()
    
#     return sum(len(re.findall(r'\b' + re.escape(neg) + r'\b', text_lower)) for neg in negations)

def negation_count(text: str) -> int:
    text = text.lower()
    text = text.replace("'", "")

    pattern = r"\b(not|no|never|nothing|none|nobody|dont|doesnt|didnt|wont|cant|cannot|isnt|arent|wasnt|werent)\b"

    return len(re.findall(pattern, text))

def word_tokenize(text: str):
    return re.findall(r'\b\w+\b', text.lower())


def syllable_count(word: str) -> int:
    """
    Simple heuristic syllable counter (no external libs).
    Good enough for relative complexity signals.
    """
    word = word.lower()
    word = re.sub(r'[^a-z]', '', word)

    if len(word) <= 3:
        return 1

    vowels = "aeiouy"
    count = 0
    prev_vowel = False

    for char in word:
        is_vowel = char in vowels
        if is_vowel and not prev_vowel:
            count += 1
        prev_vowel = is_vowel

    if word.endswith("e"):
        count = max(1, count - 1)

    return max(count, 1)


# ----------------------------
# NEW: Readability (Flesch Reading Ease)
# ----------------------------

def flesch_reading_ease(text: str) -> float:
    words = word_tokenize(text)
    sentences = count_sentences(text)

    if len(words) == 0 or sentences == 0:
        return 0.0

    syllables = sum(syllable_count(w) for w in words)

    # Flesch Reading Ease formula
    # 206.835 - 1.015*(words/sentences) - 84.6*(syllables/words)
    return 206.835 - 1.015 * (len(words) / sentences) - 84.6 * (syllables / len(words))


# ----------------------------
# NEW: Complexity Score (composite proxy)
# ----------------------------

def complexity_score(text: str) -> float:
    words = word_tokenize(text)
    sentences = count_sentences(text)

    if not words:
        return 0.0

    avg_word_len = np.mean([len(w) for w in words])
    avg_syllables = np.mean([syllable_count(w) for w in words])
    sentence_complexity = len(words) / max(sentences, 1)

    # weighted heuristic (interpretability > perfection)
    return (
        0.4 * avg_word_len +
        0.4 * avg_syllables +
        0.2 * sentence_complexity
    )


# ----------------------------
# NEW: Repetition Score
# ----------------------------

def repetition_score(text: str) -> float:
    words = word_tokenize(text)
    if not words:
        return 0.0

    total_words = len(words)
    unique_words = len(set(words))

    # lexical repetition ratio
    repetition_ratio = 1 - (unique_words / total_words)

    # consecutive repetition (e.g. "very very")
    consecutive_repeats = sum(words[i] == words[i + 1] for i in range(len(words) - 1))

    return repetition_ratio + (consecutive_repeats / (total_words + 1))


# ----------------------------
# Main Feature Pipeline
# ----------------------------

def extract_trump_style_features(df: pd.DataFrame, text_col: str = "text") -> pd.DataFrame:
    
    df = df.copy()

    # Basisfeatures
    df["tweet_length_chars"] = df[text_col].astype(str).apply(len)
    df["tweet_length_words"] = df[text_col].astype(str).apply(lambda x: len(re.findall(r'\b\w+\b', x)))
    df["sentence_count"] = df[text_col].astype(str).apply(count_sentences)
    df["avg_word_length"] = df[text_col].astype(str).apply(avg_word_length)

    # Stilmarker
    df["all_caps_words"] = df[text_col].astype(str).apply(count_all_caps_words)
    df["exclamation_count"] = df[text_col].astype(str).apply(lambda x: x.count("!"))
    df["question_count"] = df[text_col].astype(str).apply(lambda x: x.count("?"))
    df["repeated_words"] = df[text_col].astype(str).apply(count_repeated_words)

    df["capital_ratio"] = df[text_col].astype(str).apply(capital_ratio)

    df["negation_count"] = df[text_col].astype(str).apply(negation_count)

    # Superlatives / emotional words

    superlatives = [
    "best", "greatest", "worst", "biggest", "strongest", "weakest",
    "highest", "lowest", "tremendous", "tremendously", "huge",
    "massive", "enormous", "fantastic", "incredible", "amazing",
    "terrible", "horrible", "disaster", "perfect", "brilliant"
    ]

    intensifiers = [
    "very", "really", "totally", "absolutely", "so", "extremely",
    "truly", "just", "much", "far", "way", "super", "highly"
    ]

    df["superlative_count"] = df[text_col].apply(lambda x: count_pattern_words(x, superlatives))
    df["intensifier_count"] = df[text_col].apply(lambda x: count_pattern_words(x, intensifiers))

    # Normalisierte Features (wichtig für ML!)
    df["exclam_per_word"] = df["exclamation_count"] / (df["tweet_length_words"] + 1)
    

    df["readability_score"] = df[text_col].apply(flesch_reading_ease)

    df["complexity_score"] = df[text_col].apply(complexity_score)

    df["repetition_score"] = df[text_col].apply(repetition_score)


    return df

In [ ]:
df_features = extract_trump_style_features(df, 'clean_text')

# print(df_features.head())

             id                                               text  \
0  9.845497e+16  Republicans and Democrats have both created ou...   
1  1.234653e+18  I was thrilled to be back in the Great city of...   
2  1.304875e+18  The Unsolicited Mail In Ballot Scam is a major...   
3  1.223641e+18  Getting a little exercise this morning! https:...   
4  1.319502e+18                            https://t.co/4qwCKQOiOw   

               datetime        date              timestamp_utc  \
0  2011-08-02T18:07:48Z  2011-08-02  2011-08-02 18:07:48+00:00   
1  2020-03-03T01:34:50Z  2020-03-03  2020-03-03 01:34:50+00:00   
2  2020-09-12T20:10:58Z  2020-09-12  2020-09-12 20:10:58+00:00   
3  2020-02-01T16:14:02Z  2020-02-01  2020-02-01 16:14:02+00:00   
4  2020-10-23T04:52:14Z  2020-10-23  2020-10-23 04:52:14+00:00   

                                          clean_text  \
0  Republicans and Democrats have both created ou...   
1  I was thrilled to be back in the Great city of...   
2  The Unsolic

In [33]:
df_backup = df.copy()

In [37]:
df = df_features.copy()

In [39]:
df.shape

(46694, 32)

### Tweet-Historie

In [ ]:
## Burst Features
df = df.sort_values("timestamp_utc")

df["tweets_last_1h"] = df["timestamp_utc"].rolling("1h", on="timestamp_utc").count()
df["tweets_last_6h"] = df["timestamp_utc"].rolling("6h", on="timestamp_utc").count()
df["tweets_last_12h"] = df["timestamp_utc"].rolling("12h", on="timestamp_utc").count()
df["tweets_last_24h"] = df["timestamp_utc"].rolling("24h", on="timestamp_utc").count()

Alternative

In [ ]:
df = df.sort_values("timestamp_utc")

df = df.set_index("timestamp_utc")

df["tweets_last_1h"] = df.rolling("1h").size()
df["tweets_last_6h"] = df.rolling("6h").size()
df["tweets_last_12h"] = df.rolling("12h").size()
df["tweets_last_24h"] = df.rolling("24h").size()

df = df.reset_index()

## Rule-Based NLP Features

In [ ]:
# OIL_TERMS = ["oil", "barrel", "opec", "gas", "petroleum", "energy"]
# TRADE_TERMS = ["china", "tariff", "trade", "deal"]
# GEO_TERMS = ["iran", "syria", "russia", "war", "military"]
# ECON_TERMS = ["fed", "interest", "inflation", "jobs", "gdp"]

In [35]:
OIL_TERMS = [
    "oil", "crude", "crude oil", "wti", "brent", "barrel",
    "opec", "opec+", "gas", "natural gas", "lng",
    "petroleum", "energy",
    "refinery", "refining", "drilling", "rig", "rigs",
    "production", "output", "supply", "demand",
    "inventories", "stockpile", "storage",
    "spr", "strategic petroleum reserve",
    "inventory", "exports", "imports",
    "futures", "commodity", "commodities"
]

TRADE_TERMS = [
    "china", "tariff", "trade", "deal", "agreement",
    "tariffs", "sanctions", "embargo",
    "nafta", "usmca", "eu", "europe",
    "mexico", "canada",
    "trade war", "currency manipulation"
]

GEO_TERMS = [
    "iran", "iraq", "syria", "russia", "venezuela",
    "saudi", "saudi arabia", "libya",
    "war", "military", "conflict",
    "attack", "strike", "bombing",
    "middle east", "geopolitical"
]

ECON_TERMS = [
    "fed", "federal reserve", "interest", "interest rate",
    "inflation", "cpi", "pce",
    "jobs", "unemployment", "nonfarm payrolls", "nfp",
    "gdp", "growth",
    "recession",
    "dollar", "usd",
    "yield", "treasury",
    "quantitative easing", "qe",
    "monetary policy",
    "stimulus", "fiscal", "budget"
]

# Optional (sehr relevant für Trump-Tweets als Kontextsignal)
# TRUMP_CONTEXT_TERMS = [
#     "trump", "donald trump", "president", "white house",
#     "administration"
# ]

In [36]:
def keyword_flag(text, keywords):
    return int(any(k in text for k in keywords))


df["mentions_oil"] = df["clean_text"].apply(lambda x: keyword_flag(x, OIL_TERMS))
df["mentions_trade"] = df["clean_text"].apply(lambda x: keyword_flag(x, TRADE_TERMS))
df["mentions_geo"] = df["clean_text"].apply(lambda x: keyword_flag(x, GEO_TERMS))
df["mentions_econ"] = df["clean_text"].apply(lambda x: keyword_flag(x, ECON_TERMS))

## Stylometric Features (Trump Signatur)

In [38]:
df["tweet_length"] = df["clean_text"].str.len()
df["word_count"] = df["clean_text"].str.split().str.len()

df["exclamation_count"] = df["text"].str.count("!")
df["question_count"] = df["text"].str.count(r"\?")

df["caps_ratio"] = df["text"].apply(
    lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1)
)

## Sentiment FinBERT

In [47]:
sentiment_model = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [48]:
def get_sentiment(text):
    try:
        res = sentiment_model(text[:512])[0]
        label = res["label"]
        score = res["score"]

        if label == "positive":
            return score
        elif label == "negative":
            return -score
        else:
            return 0
    except:
        return 0


df["sentiment_score"] = df["clean_text"].apply(get_sentiment)

## Emotion Proxy

In [49]:
POS_WORDS = ["great", "strong", "win", "good", "amazing"]
NEG_WORDS = ["bad", "terrible", "weak", "crisis", "fake"]

def emotion_score(text):
    pos = sum(w in text for w in POS_WORDS)
    neg = sum(w in text for w in NEG_WORDS)
    return pos - neg


df["emotion_score"] = df["clean_text"].apply(emotion_score)

## Transformer Embedding (Sentence-BERT)

In [51]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\02_Florian_Benutzer\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [52]:
embeddings = embed_model.encode(
    df["clean_text"].tolist(),
    show_progress_bar=True
)

embeddings_df = pd.DataFrame(
    embeddings,
    columns=[f"emb_{i}" for i in range(embeddings.shape[1])]
)

df = pd.concat([df.reset_index(drop=True), embeddings_df], axis=1)

Batches:   0%|          | 0/1460 [00:00<?, ?it/s]

## Optional: PCA reduzieren (empfohlen)

In [53]:
from sklearn.decomposition import PCA

pca = PCA(n_components=30)
emb_pca = pca.fit_transform(embeddings)

emb_pca_df = pd.DataFrame(
    emb_pca,
    columns=[f"emb_pca_{i}" for i in range(30)]
)

df = pd.concat([df, emb_pca_df], axis=1)

### Alternative

In [57]:
df.set_index("timestamp_utc", inplace=True)

df["tweets_last_1h"] = df["text"].rolling("1h").count()
df["tweets_last_6h"] = df["text"].rolling("6h").count()

df = df.reset_index()

In [58]:
df

,timestamp_utc,id,text,datetime,date,clean_text,hour,day_of_week,is_weekend,minute,...,emb_pca_22,emb_pca_23,emb_pca_24,emb_pca_25,emb_pca_26,emb_pca_27,emb_pca_28,emb_pca_29,tweets_last_1h,tweets_last_6h
0,2009-05-04 18:54:25+00:00,1.698309e+09,Be sure to tune in and watch Donald Trump on L...,2009-05-04T18:54:25Z,2009-05-04,be sure to tune in and watch donald trump on l...,18,0,0,54,...,0.147628,-0.120222,-0.003056,-0.042803,0.012091,0.036382,-0.011314,0.060289,1.0,1.0
1,2009-05-05 01:00:10+00:00,1.701461e+09,Donald Trump will be appearing on The View tom...,2009-05-05T01:00:10Z,2009-05-05,donald trump will be appearing on the view tom...,1,1,0,0,...,0.115513,-0.006284,-0.036056,-0.039293,0.012569,0.070970,-0.112187,-0.020383,1.0,1.0
2,2009-05-08 13:38:08+00:00,1.737480e+09,Donald Trump reads Top Ten Financial Tips on L...,2009-05-08T13:38:08Z,2009-05-08,donald trump reads top ten financial tips on l...,13,4,0,38,...,0.098122,-0.128410,0.036980,0.003709,0.037471,0.052561,-0.012630,0.043487,1.0,1.0
3,2009-05-08 20:40:15+00:00,1.741161e+09,New Blog Post: Celebrity Apprentice Finale and...,2009-05-08T20:40:15Z,2009-05-08,new blog post: celebrity apprentice finale and...,20,4,0,40,...,-0.078529,-0.028899,-0.093646,0.007925,0.095469,0.240477,-0.061366,-0.045210,1.0,1.0
4,2009-05-12 14:07:28+00:00,1.773561e+09,"""""""My persona will never be that of a wallflow...",2009-05-12T14:07:28Z,2009-05-12,"""""""my persona will never be that of a wallflow...",14,1,0,7,...,0.063643,0.019768,-0.022095,-0.148142,0.037149,-0.065220,0.079894,-0.006261,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46689,2021-01-06 21:17:24+00:00,1.346929e+18,https://t.co/Pm2PKV0Fp3,2021-01-06T21:17:24Z,2021-01-06,,21,2,0,17,...,0.023748,-0.017323,0.008929,-0.005868,-0.007515,0.021483,-0.006002,0.005868,1.0,6.0
46690,2021-01-06 23:01:04+00:00,1.346955e+18,These are the things and events that happen wh...,2021-01-06T23:01:04Z,2021-01-06,these are the things and events that happen wh...,23,2,0,1,...,-0.026220,0.126473,0.006688,-0.130505,-0.044662,0.097333,-0.080510,0.112731,1.0,6.0
46691,2021-01-08 00:10:24+00:00,1.347335e+18,https://t.co/csX07ZVWGe,2021-01-08T00:10:24Z,2021-01-08,,0,4,0,10,...,0.023748,-0.017323,0.008929,-0.005868,-0.007515,0.021483,-0.006002,0.005868,1.0,1.0
46692,2021-01-08 14:46:38+00:00,1.347555e+18,"The 75,000,000 great American Patriots who vot...",2021-01-08T14:46:38Z,2021-01-08,"the 75,000,000 great american patriots who vot...",14,4,0,46,...,0.102513,0.042074,0.029353,-0.151952,-0.137809,-0.050694,0.039298,-0.023867,1.0,1.0


In [59]:
df.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\df.csv", sep=",", index=False, encoding="utf-8")

## Novelty Feature

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df["clean_text"])

df["novelty_score"] = np.asarray(X_tfidf.mean(axis=1)).ravel()

### seltene Begriffe

## Interaction Features

In [ ]:
df["sentiment_x_oil"] = df["sentiment_score"] * df["mentions_oil"]
df["emotion_x_caps"] = df["emotion_score"] * df["caps_ratio"]
df["geo_x_exclamation"] = df["mentions_geo"] * df["exclamation_count"]

## Final Feature Matrix

In [ ]:
feature_cols = [
    "sentiment_score",
    "emotion_score",
    "mentions_oil",
    "mentions_trade",
    "mentions_geo",
    "mentions_econ",
    "tweet_length",
    "word_count",
    "caps_ratio",
    "exclamation_count",
    "question_count",
    "tweets_last_1h",
    "tweets_last_6h",
    "novelty_score",
]

X = pd.concat([df[feature_cols], df.filter(like="emb_pca_")], axis=1)

## RoBERTA

Für Emotion Detection ist das beste Standardmodell:

👉 cardiffnlp/twitter-roberta-base-emotion

Es klassifiziert typischerweise in:

anger
joy
optimism
sadness

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import pandas as pd

### Modell laden

In [ ]:
model_name = "cardiffnlp/twitter-roberta-base-emotion"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

emotion_pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True
)

### Emotion Feature Engineering

Wir extrahieren nicht nur Label, sondern vollständige Score-Verteilung → viel besser für ML.

In [ ]:
def get_emotion_scores(text):
    try:
        results = emotion_pipe(text[:512])[0]
        
        scores = {r["label"]: r["score"] for r in results}
        
        return pd.Series({
            "emotion_anger": scores.get("anger", 0),
            "emotion_joy": scores.get("joy", 0),
            "emotion_optimism": scores.get("optimism", 0),
            "emotion_sadness": scores.get("sadness", 0),
        })
    
    except:
        return pd.Series({
            "emotion_anger": 0,
            "emotion_joy": 0,
            "emotion_optimism": 0,
            "emotion_sadness": 0,
        })

### Apply auf DataFrame

In [ ]:
emotion_features = df["clean_text"].apply(get_emotion_scores)

df = pd.concat([df, emotion_features], axis=1)

### Emotion Features ableiten

In [ ]:
df["emotion_negative"] = df["emotion_anger"] + df["emotion_sadness"]
df["emotion_positive"] = df["emotion_joy"] + df["emotion_optimism"]

df["emotion_net"] = df["emotion_positive"] - df["emotion_negative"]

### Market-relevante Transformationsfeatures

In [ ]:
df["emotion_intensity"] = df[
    ["emotion_anger", "emotion_joy", "emotion_optimism", "emotion_sadness"]
].max(axis=1)

df["emotion_conflict"] = (
    df["emotion_positive"] * df["emotion_negative"]
)

Interpretation:

hoher emotion_conflict → ambivalente, unklare Tweets → oft noisy market signal

## Optional: Rolling Emotion Shock (sehr stark!)

In [ ]:
df = df.sort_values("timestamp")

df["emotion_net_ma_24h"] = df["emotion_net"].rolling("24h", on=df["timestamp"]).mean()
df["emotion_shock"] = df["emotion_net"] - df["emotion_net_ma_24h"]

Das ist extrem wichtig:

Märkte reagieren nicht auf Emotionen an sich
sondern auf Abweichungen vom Normalzustand

### Final Feature Set

In [ ]:
emotion_cols = [
    "emotion_anger",
    "emotion_joy",
    "emotion_optimism",
    "emotion_sadness",
    "emotion_negative",
    "emotion_positive",
    "emotion_net",
    "emotion_intensity",
    "emotion_conflict",
    "emotion_shock",
]